# Reviewer Response — Comment 10: Threshold-Specific Performance Metrics

**Reviewer concern (Reviewer #3):**
> The AUROC of 0.87 for mortality appears strong, but balanced accuracy is only 0.63
> and the Brier score of 0.043 is close to the baseline of ~0.043 one would achieve by
> predicting the prevalence for every patient. The authors should report sensitivity,
> specificity, positive predictive value, and negative predictive value at clinically
> relevant thresholds. The net benefit of 1.5 additional patients per 100 at a 10%
> threshold, while positive, is modest and confidence intervals are not provided.

**Our approach:**
1. Re-train the primary Random Forest on the same 70/30 split used in the main analysis.
2. Report sensitivity, specificity, PPV, NPV, LR+, LR− at clinical thresholds: 5%, 10%, 20%.
3. Address Brier-vs-baseline argument with a proper null-model Brier comparison.
4. Net benefit CIs already computed in Comment 4 — cite here.


In [ ]:
import os, warnings
os.environ['MPLCONFIGDIR'] = '/tmp/mpl_cache'
import matplotlib
matplotlib.use('Agg')

_here = os.getcwd()
if os.path.exists(os.path.join(_here, 'model_combined_dataset_clustered_withmissingness.csv')):
    INSPIRE_ROOT = _here
    OUTPUT_DIR   = _here
else:
    INSPIRE_ROOT = os.path.normpath(os.path.join(_here, '..', '..'))
    OUTPUT_DIR   = _here
os.chdir(INSPIRE_ROOT)
print('Root:', INSPIRE_ROOT)
print('Output:', OUTPUT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, brier_score_loss,
                              confusion_matrix, roc_curve)
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('model_combined_dataset_clustered_withmissingness.csv')
print(f'Dataset shape: {df.shape}')

EXCLUDE = ['subject_id','op_id','cluster','died_inhospital',
           'inhosp_death_time','icuin_time','icu_long']
exclude_present = [c for c in EXCLUDE if c in df.columns]
feat_cols = [c for c in df.columns if c not in exclude_present]
print(f'Feature columns: {len(feat_cols)}')

# Derive ICU-long outcome (>3 days) if not present
if 'icu_long' not in df.columns:
    if 'icuin_time' in df.columns and 'opend_time' in df.columns:
        df['icu_long'] = ((pd.to_datetime(df['icuin_time']) -
                           pd.to_datetime(df['opend_time'])).dt.total_seconds()
                          / 3600 / 24 > 3).astype(int)
    else:
        # Fall back: look for any icu-stay column
        icu_col = [c for c in df.columns if 'icu' in c.lower() and 'long' in c.lower()]
        if icu_col:
            df['icu_long'] = df[icu_col[0]]
        else:
            # Try from patient_timeline
            df['icu_long'] = np.nan

print(f"Mortality prevalence: {df['died_inhospital'].mean():.3f}")
if 'icu_long' in df.columns and df['icu_long'].notna().any():
    print(f"ICU >3d prevalence: {df['icu_long'].mean():.3f}")

In [ ]:
# ── Helper: threshold table ────────────────────────────────────────────────────
def threshold_table(y_true, y_prob, thresholds, label):
    rows = []
    prev = y_true.mean()
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        sens  = tp / (tp + fn) if (tp+fn) > 0 else np.nan
        spec  = tn / (tn + fp) if (tn+fp) > 0 else np.nan
        ppv   = tp / (tp + fp) if (tp+fp) > 0 else np.nan
        npv   = tn / (tn + fn) if (tn+fn) > 0 else np.nan
        lrp   = sens / (1 - spec) if (1-spec) > 0 else np.nan
        lrn   = (1 - sens) / spec if spec > 0 else np.nan
        nb    = (tp/len(y_true)) - (fp/len(y_true)) * (t/(1-t))  # net benefit
        rows.append({
            'Outcome': label,
            'Threshold': f'{t:.0%}',
            'Sensitivity': f'{sens:.3f}',
            'Specificity': f'{spec:.3f}',
            'PPV': f'{ppv:.3f}',
            'NPV': f'{npv:.3f}',
            'LR+': f'{lrp:.2f}',
            'LR-': f'{lrn:.3f}',
            'Net Benefit': f'{nb:.4f}',
            'N predicted positive': int(tp+fp),
            'True positives': int(tp),
        })
    return pd.DataFrame(rows)


# ── Fit RF: mortality ──────────────────────────────────────────────────────────
y_mort = df['died_inhospital'].values
X = df[feat_cols].fillna(0).values

X_tr, X_te, y_tr, y_te = train_test_split(X, y_mort, test_size=0.3,
                                            random_state=42, stratify=y_mort)
try:
    sm = SMOTE(random_state=42)
    X_tr_s, y_tr_s = sm.fit_resample(X_tr, y_tr)
except Exception:
    X_tr_s, y_tr_s = X_tr, y_tr

rf_mort = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_mort.fit(X_tr_s, y_tr_s)
prob_mort = rf_mort.predict_proba(X_te)[:, 1]

auroc_mort = roc_auc_score(y_te, prob_mort)
brier_mort = brier_score_loss(y_te, prob_mort)
brier_null_mort = brier_score_loss(y_te, np.full_like(prob_mort, y_tr.mean()))
print(f'Mortality AUROC: {auroc_mort:.3f}')
print(f'Mortality Brier: {brier_mort:.4f}  |  Null-model Brier: {brier_null_mort:.4f}')
print(f'Brier Skill Score (1 - model/null): {1 - brier_mort/brier_null_mort:.3f}')

In [ ]:
# ── Fit RF: ICU extended ───────────────────────────────────────────────────────
icu_col_name = None
for c in df.columns:
    if 'icu' in c.lower() and ('long' in c.lower() or 'extend' in c.lower() or '>3' in c.lower()):
        icu_col_name = c; break

results_icu = None
if icu_col_name and df[icu_col_name].notna().sum() > 100:
    y_icu = df[icu_col_name].dropna()
    X_icu = df.loc[y_icu.index, feat_cols].fillna(0).values
    y_icu = y_icu.values
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_icu, y_icu, test_size=0.3,
                                                    random_state=42, stratify=y_icu)
    try:
        X_tr2_s, y_tr2_s = SMOTE(random_state=42).fit_resample(X_tr2, y_tr2)
    except Exception:
        X_tr2_s, y_tr2_s = X_tr2, y_tr2
    rf_icu = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_icu.fit(X_tr2_s, y_tr2_s)
    prob_icu = rf_icu.predict_proba(X_te2)[:, 1]
    auroc_icu = roc_auc_score(y_te2, prob_icu)
    brier_icu = brier_score_loss(y_te2, prob_icu)
    brier_null_icu = brier_score_loss(y_te2, np.full_like(prob_icu, y_tr2.mean()))
    print(f'ICU extended AUROC: {auroc_icu:.3f}')
    print(f'ICU extended Brier: {brier_icu:.4f}  |  Null Brier: {brier_null_icu:.4f}')
    print(f'ICU Brier Skill Score: {1 - brier_icu/brier_null_icu:.3f}')
    results_icu = (y_te2, prob_icu, y_tr2.mean(), brier_icu, brier_null_icu)
else:
    print('ICU-long column not found or too few observations — mortality only.')

In [ ]:
# ── Threshold tables ──────────────────────────────────────────────────────────
THRESHOLDS = [0.05, 0.10, 0.20]

df_mort_tbl = threshold_table(y_te, prob_mort, THRESHOLDS, 'Mortality')
print('\n=== Mortality — threshold performance ===')
print(df_mort_tbl.to_string(index=False))

if results_icu:
    y_te2, prob_icu, *_ = results_icu
    df_icu_tbl = threshold_table(y_te2, prob_icu, THRESHOLDS, 'ICU >3 days')
    print('\n=== ICU extended — threshold performance ===')
    print(df_icu_tbl.to_string(index=False))
    combined = pd.concat([df_mort_tbl, df_icu_tbl], ignore_index=True)
else:
    combined = df_mort_tbl

combined.to_csv(os.path.join(OUTPUT_DIR, 'threshold_performance_table.csv'), index=False)
print('\nSaved: threshold_performance_table.csv')

In [ ]:
# ── Brier Skill Score explanation table ───────────────────────────────────────
bss_data = {
    'Outcome': ['Mortality', 'ICU >3 days'] if results_icu else ['Mortality'],
    'Prevalence': [
        f"{y_tr.mean():.3f}",
        f"{results_icu[2]:.3f}" if results_icu else None
    ],
    'Model Brier': [
        f"{brier_mort:.4f}",
        f"{results_icu[3]:.4f}" if results_icu else None
    ],
    'Null-model Brier': [
        f"{brier_null_mort:.4f}",
        f"{results_icu[4]:.4f}" if results_icu else None
    ],
    'Brier Skill Score': [
        f"{1 - brier_mort/brier_null_mort:.3f}",
        f"{1 - results_icu[3]/results_icu[4]:.3f}" if results_icu else None
    ]
}
df_bss = pd.DataFrame(bss_data).dropna()
print(df_bss.to_string(index=False))
df_bss.to_csv(os.path.join(OUTPUT_DIR, 'brier_skill_score.csv'), index=False)
print('Saved: brier_skill_score.csv')

In [ ]:
# ── Figure: ROC with threshold markers ────────────────────────────────────────
fig, axes = plt.subplots(1, 2 if results_icu else 1, figsize=(10 if results_icu else 5, 4.5))
if not results_icu:
    axes = [axes]

def plot_roc_with_thresholds(ax, y_true, y_prob, thresholds, title, color='steelblue'):
    fpr, tpr, roc_thresh = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'RF (AUC = {auc:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5)
    markers = ['o','s','^']
    colors_t = ['#e74c3c','#e67e22','#8e44ad']
    for i, t in enumerate(thresholds):
        y_pred = (y_prob >= t).astype(int)
        tp = ((y_pred==1)&(y_true==1)).sum()
        fp = ((y_pred==1)&(y_true==0)).sum()
        fn = ((y_pred==0)&(y_true==1)).sum()
        tn = ((y_pred==0)&(y_true==0)).sum()
        sens_t = tp/(tp+fn) if (tp+fn)>0 else 0
        spec_t = tn/(tn+fp) if (tn+fp)>0 else 0
        ax.plot(1-spec_t, sens_t, markers[i], color=colors_t[i],
                markersize=10, label=f'Threshold {t:.0%}', zorder=5)
    ax.set_xlabel('1 − Specificity', fontsize=10)
    ax.set_ylabel('Sensitivity', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.grid(True, alpha=0.3)

plot_roc_with_thresholds(axes[0], y_te, prob_mort, THRESHOLDS,
                         'Mortality Model — ROC with clinical thresholds')
if results_icu:
    y_te2, prob_icu, *_ = results_icu
    plot_roc_with_thresholds(axes[1], y_te2, prob_icu, THRESHOLDS,
                             'ICU >3d Model — ROC with clinical thresholds',
                             color='#27ae60')

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'roc_with_thresholds.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: roc_with_thresholds.png')

## Summary for Reviewer Response

### Addressing Reviewer #3's specific concerns

**1. Balanced accuracy = 0.63 for mortality**
Balanced accuracy averages sensitivity and specificity at the default 0.5 probability
threshold — which is uninformative for a 4.5%-prevalence outcome where no clinician would
intervene only above 50% predicted risk. The threshold-specific table above provides
sensitivity, specificity, PPV, NPV, and net benefit at the **5%, 10%, and 20%** risk cutoffs
that are operationally relevant for perioperative triage decisions.

**2. Brier score ≈ null-model Brier**
The Brier Skill Score (BSS = 1 − Brier_model / Brier_null) provides the appropriate
reference: BSS > 0 indicates the model improves on the null (predict-prevalence) baseline.
The BSS for mortality is reported in `brier_skill_score.csv`.

**3. Net benefit CI**
Bootstrap 95% CIs for net benefit are computed in `comment_4_calibration/dca_bootstrap_ci.csv`
and plotted in `dca_with_bootstrap_ci.png`. The threshold performance table above
reports point-estimate net benefit at each threshold consistent with the DCA analysis.

### Proposed manuscript amendments
- **Results:** Add threshold performance table (sens/spec/PPV/NPV) as Supplementary Table.
- **Results:** Add Brier Skill Score alongside raw Brier in Table 2.
- **Methods:** Add one sentence: "Threshold-specific performance (sensitivity, specificity,
  PPV, NPV) was evaluated at 5%, 10%, and 20% predicted risk cutoffs; Brier Skill Score
  (1 − Brier_model / Brier_null) was used to contextualise absolute Brier scores relative
  to a prevalence-only null model."
